# 01 · Preprocesamiento del corpus del proyecto

**Taller de una hora.** Al terminar tenéis que haber tomado cuatro decisiones y
haberlas anotado en `docs/bitacora.md`:

1. Qué tokenizador usáis.
2. Si pasáis a minúsculas y en qué momento.
3. Qué hacéis con las stopwords.
4. Stemming o lematización.

El Hito 1 no pide el preprocesamiento *aplicado*, pide el preprocesamiento
**justificado**. Este notebook existe para que podáis justificarlo con datos
de vuestro propio corpus, no con opiniones.

## 0 · Preparación

Ejecutad esta celda una vez. Tarda un par de minutos la primera vez.

In [ ]:
%pip install -q nltk spacy pandas

import importlib.util, subprocess, sys

# El modelo de espanol no viene con spacy: hay que descargarlo una vez.
if importlib.util.find_spec('es_core_news_sm') is None:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'], check=True)

import nltk
for recurso in ['punkt_tab', 'stopwords']:
    nltk.download(recurso, quiet=True)

print('entorno listo')

## 1 · Cargar el corpus

Poned vuestro fichero en `data/raw/` y cambiad la ruta. Si todavía no tenéis datos,
la celda usa un corpus de ejemplo para que podáis seguir el taller.

In [ ]:
import pandas as pd
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUTA = raiz / 'data' / 'raw' / 'corpus.csv'   # <-- cambiad esto

if RUTA.exists():
    df = pd.read_csv(RUTA, encoding='utf-8')
    print(f'Corpus propio: {len(df)} documentos')
else:
    df = pd.DataFrame({'texto': [
        'No me gustó NADA el envío, llegó 3 días tarde 😡',
        '¿Por qué no avisan cuando hay retraso? Llevo dos semanas esperando.',
        'Atención al cliente impecable, me lo resolvieron en el momento.',
        'El producto está bien pero el embalaje venía roto.',
        'Nunca volveré a comprar aquí. Un desastre de principio a fin.',
        'Buenísimo, superó mis expectativas. Repetiré sin duda.',
    ]})
    print('AVISO: sin corpus propio, se usa el de ejemplo.')

COL = 'texto'   # <-- nombre de vuestra columna de texto
df.head()

## 2 · Mirar el texto crudo antes de tocarlo

Primera regla: mirad los datos. Longitudes, caracteres raros, duplicados, vacíos.

In [ ]:
s = df[COL].astype(str)
print(f'documentos      : {len(s)}')
print(f'vacíos          : {(s.str.strip() == "").sum()}')
print(f'duplicados      : {s.duplicated().sum()}')
print(f'longitud media  : {s.str.len().mean():.0f} caracteres')
print(f'longitud mín/máx: {s.str.len().min()} / {s.str.len().max()}')

import re
raros = set(re.findall(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ.,;:¿?¡!()\-\'\"]', ' '.join(s)))
print(f'caracteres no habituales: {sorted(raros)[:20]}')

**Decisión 0.** ¿Elimináis duplicados y vacíos? Casi siempre sí, pero anotad cuántos
eran: si el 30% de vuestro corpus estaba duplicado, eso dice algo de la fuente.

## 3 · Tokenizar: dos herramientas, dos resultados

Aquí está la primera decisión real. Comparad NLTK y spaCy **sobre vuestro texto**.

In [ ]:
from nltk.tokenize import word_tokenize
import spacy

nlp = spacy.load('es_core_news_sm')
ejemplo = s.iloc[1] if len(s) > 1 else s.iloc[0]
print('TEXTO :', ejemplo)
print()
print('NLTK  :', word_tokenize(ejemplo, language='spanish'))
print()
print('spaCy :', [t.text for t in nlp(ejemplo)])

**Mirad la diferencia con los signos de apertura.** Con `¿Por qué no avisan...`,
NLTK devuelve `'¿Por'` como un único token y spaCy separa `'¿'` y `'Por'`.

No es un detalle cosmético: si `¿Por` queda pegado, esa forma no coincidirá con `por`
en ningún recuento ni en ningún vocabulario, y habréis creado un token fantasma.

**Decisión 1.** ¿Qué tokenizador usáis? Anotad el motivo, con el ejemplo concreto de
vuestro corpus que os haya hecho decidir.

## 4 · Minúsculas: por qué el orden importa

Pasar a minúsculas parece inocuo. No lo es, y se ve mejor con un ejemplo.

In [ ]:
prueba = 'No me gustó NADA el envío'

print('%-10s %-10s %-8s' % ('TOKEN', 'LEMA', 'POS'))
for t in nlp(prueba):
    print('%-10s %-10s %-8s' % (t.text, t.lemma_, t.pos_))
print()
print('--- el mismo texto ya en minúsculas ---')
print('%-10s %-10s %-8s' % ('TOKEN', 'LEMA', 'POS'))
for t in nlp(prueba.lower()):
    print('%-10s %-10s %-8s' % (t.text, t.lemma_, t.pos_))

`NADA` en mayúsculas se etiqueta como nombre propio y su lema se queda en `NADA`.
En minúsculas se etiqueta como adverbio y lematiza a `nada`.

Es decir: **si lematizáis después de pasar a minúsculas obtenéis un resultado distinto
que si lo hacéis antes.** Y en reseñas las mayúsculas enfáticas son frecuentes.

**Decisión 2.** ¿Pasáis a minúsculas? ¿Antes o después de lematizar? Si vuestro caso
depende del énfasis —y en sentimiento suele depender—, quizá convenga guardar una
columna con el texto original antes de bajarlo todo a minúsculas.

## 5 · Stopwords: las dos listas no coinciden

Comparad la lista de NLTK con la marca `is_stop` de spaCy sobre vuestro propio texto.

In [ ]:
from nltk.corpus import stopwords

sw_nltk = set(stopwords.words('spanish'))
doc = nlp(ejemplo.lower())

print('%-14s %-10s %-10s' % ('TOKEN', 'NLTK', 'spaCy'))
for t in doc:
    if t.is_punct or t.is_space:
        continue
    print('%-14s %-10s %-10s' % (t.text,
                                 'stopword' if t.text in sw_nltk else '-',
                                 'stopword' if t.is_stop else '-'))
print()
print(f'tamaño de la lista de NLTK : {len(sw_nltk)}')
print(f'negaciones en esa lista    : """{[w for w in ["no","ni","nada","sin","nunca","tampoco","jamás"] if w in sw_nltk]}"""')
print(f'negaciones que NO están    : {[w for w in ["no","ni","nada","sin","nunca","tampoco","jamás"] if w not in sw_nltk]}')

Dos problemas a la vista.

El primero: **las dos herramientas no coinciden** en qué es una stopword. La lista de
spaCy para español marca como stopwords palabras con contenido, y la de NLTK no.

El segundo, más grave para vosotros: la lista de NLTK contiene `no`, `ni`, `nada` y
`sin`, pero **no** contiene `nunca`, `tampoco` ni `jamás`. Filtrando a ciegas,
*"no me gustó nada"* se convierte en *"gustó"* —sentido invertido— mientras
*"nunca volveré"* se queda intacto. El tratamiento de la negación queda incoherente.

**Decisión 3.** ¿Filtráis stopwords? Si vuestro proyecto tiene que ver con sentimiento,
lo razonable es partir de la lista y **retirar de ella las partículas de negación**.
La celda siguiente lo hace.

In [ ]:
NEGACIONES = {'no', 'ni', 'nada', 'sin', 'nunca', 'tampoco', 'jamás', 'nadie', 'ninguno'}
sw_propia = sw_nltk - NEGACIONES

print(f'lista original : {len(sw_nltk)}')
print(f'lista propia   : {len(sw_propia)}  (se han conservado las negaciones)')
print()
frase = 'no me gustó nada el envío'
print('con la lista original:', [w for w in frase.split() if w not in sw_nltk])
print('con la lista propia  :', [w for w in frase.split() if w not in sw_propia])

## 6 · Stemming frente a lematización

La comparación que pide el taller. Hacedla **sobre diez frases de vuestro corpus**,
no sobre el ejemplo.

In [ ]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer('spanish')
filas = []
for t in nlp(' '.join(s.head(10).tolist()).lower()):
    if t.is_punct or t.is_space or t.like_num:
        continue
    filas.append({'token': t.text, 'stem': stemmer.stem(t.text),
                  'lema': t.lemma_, 'pos': t.pos_})

comp = pd.DataFrame(filas).drop_duplicates('token')
distintos = comp[comp['stem'] != comp['lema']]
print(f'{len(comp)} tokens distintos · el stem y el lema difieren en {len(distintos)}')
distintos.head(25)

Mirad las filas donde difieren y juzgad cuál os sirve.

- El **stemmer** corta por reglas: rápido, pero produce formas que no son palabras
  (`envío` → `envi`, `días` → `dias`). Agrupa bien, se lee mal.
- El **lematizador** consulta un modelo: devuelve palabras reales (`llegó` → `llegar`),
  pero es más lento y **se equivoca**. Buscad en la tabla algún lema que no exista en
  español: los encontraréis, sobre todo en formas con pronombre pegado.

**Decisión 4.** Stem o lema. Si vuestro resultado se va a presentar a alguien —una nube
de términos, una lista de causas de queja— el lema se lee y el stem no. Si solo vais a
contar y agrupar, el stem basta y es más rápido.

## 7 · Aplicar el pipeline elegido a todo el corpus

Poned aquí vuestras cuatro decisiones. Esta función **es** vuestro preprocesamiento:
es lo que tenéis que poder explicar en el Hito 1.

In [ ]:
TOKENIZADOR = 'spaCy es_core_news_sm'   # <-- vuestra decisión 1
USAR_MINUSCULAS = True
FILTRAR_STOPWORDS = True
CONSERVAR_NEGACIONES = True
MODO = 'lema'          # 'lema' o 'stem'
MIN_LONGITUD = 2

lista_sw = sw_propia if CONSERVAR_NEGACIONES else sw_nltk

def preprocesar(texto):
    doc = nlp(texto.lower() if USAR_MINUSCULAS else texto)
    salida = []
    for t in doc:
        if t.is_punct or t.is_space or t.like_url or t.like_email:
            continue
        pieza = t.lemma_ if MODO == 'lema' else stemmer.stem(t.text)
        pieza = pieza.lower()
        if len(pieza) < MIN_LONGITUD:
            continue
        if FILTRAR_STOPWORDS and pieza in lista_sw:
            continue
        salida.append(pieza)
    return salida

df['tokens'] = s.apply(preprocesar)
df['n_tokens'] = df['tokens'].apply(len)
df[[COL, 'tokens']].head()

## 8 · Qué ha cambiado

Comprobad el efecto antes de dar el preprocesamiento por bueno.

In [ ]:
from collections import Counter

crudo = Counter(' '.join(s).lower().split())
limpio = Counter(t for lista in df['tokens'] for t in lista)

print(f'vocabulario antes  : {len(crudo)}')
print(f'vocabulario después: {len(limpio)}')
print(f'tokens antes       : {sum(crudo.values())}')
print(f'tokens después     : {sum(limpio.values())}')
print()
print('15 términos más frecuentes tras el preprocesamiento:')
for palabra, n in limpio.most_common(15):
    print(f'  {palabra:<18} {n}')

**Comprobación de sentido.** Si entre los quince términos más frecuentes aparecen
palabras sin contenido, la lista de stopwords se queda corta para vuestro dominio.
Si falta algo que esperabais ver, algún paso se lo ha comido. Las dos cosas son
hallazgos que van a la bitácora.

### Qué se ha perdido por el camino

Cada paso del pipeline **destruye información**. Comprobad exactamente qué ha desaparecido.

In [ ]:
for i in range(min(3, len(df))):
    print(f'--- documento {i} ---')
    print('  antes  :', str(s.iloc[i])[:70])
    print('  después:', df['tokens'].iloc[i])
    print()

Mirad el primer documento del corpus de ejemplo: entra
`'No me gustó NADA el envío, llegó 3 días tarde 😡'` y sale
`['no','gustar','nada','envío','llegar','día','tarde']`.

**Han desaparecido el `3` y el emoji**, y ninguno de los dos estaba en vuestra lista de
stopwords: los ha eliminado el filtro `MIN_LONGITUD = 2`, porque ambos son de un solo
carácter. Nadie lo decidió; fue un efecto colateral.

Si vuestro proyecto es de sentimiento, el emoji era probablemente la señal más clara del
documento. Y si os interesan los plazos de entrega, `3 días` era el dato. Antes de dar el
preprocesamiento por bueno, decidid **a propósito** qué hacéis con los números y con los
emojis: conservarlos, sustituirlos por una etiqueta (`<NUM>`, `<EMOJI>`) o eliminarlos.

Es la lección general del taller: **un paso de preprocesamiento que no habéis justificado
está borrando datos sin que lo sepáis.**

## 9 · Guardar y documentar

In [ ]:
destino = raiz / 'data' / 'clean' / 'corpus_preprocesado.csv'
destino.parent.mkdir(parents=True, exist_ok=True)
guardar = df.copy()
guardar['tokens'] = guardar['tokens'].apply(lambda x: ' '.join(x))
guardar.to_csv(destino, index=False, encoding='utf-8')
print('guardado en', destino.relative_to(raiz))

print()
print('--- copiad esto en docs/bitacora.md ---')
print(f'Tokenizador          : {TOKENIZADOR}')
print(f'Minúsculas           : {USAR_MINUSCULAS}')
print(f'Filtrado de stopwords: {FILTRAR_STOPWORDS} (negaciones conservadas: {CONSERVAR_NEGACIONES})')
print(f'Normalización        : {MODO}')
print(f'Longitud mínima      : {MIN_LONGITUD}')
print(f'Vocabulario resultante: {len(limpio)} términos sobre {len(df)} documentos')

---

## Antes de cerrar

- [ ] `data/clean/corpus_preprocesado.csv` existe.
- [ ] Las cuatro decisiones están anotadas en `docs/bitacora.md`, **con el motivo**.
- [ ] Para cada decisión hay un ejemplo de vuestro corpus que la respalda.
- [ ] El notebook está confirmado en el repositorio.

Recordad que `data/clean/` no se sube al repositorio: lo que se sube es el notebook y
la bitácora. Si queréis que el resultado sea reproducible, dejad una muestra pequeña
en `data/sample/`.